# Using `pyologger` data processing pipeline with `DiveDB`
Uses classes `Metadata` and `DataReader` to facilitate data intake, processing, and alignment. 

## Read deployment metadata

In [ ]:
import re
# Import pyologger utilities
from pyologger.utils.folder_manager import *
from pyologger.plot_data.plotter import *
from pyologger.utils.param_manager import ParamManager
from pyologger.load_data.datareader import DataReader
from pyologger.load_data.metadata import Metadata

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()

In [ ]:
data_dir

### Fetch metadata

Load in metadata stored in Notion databases. Alternatively, load in your own metadata in separate dataframes for deployments, loggers, recordings, animals, and datasets. See examples here in the `metadata_snapshot.pkl` file.

In [ ]:
import os
import json
import pickle
import tempfile
from datetime import datetime, timedelta

# ---- user-provided context assumed ----
# data_dir: base data directory
# config: dict with paths (e.g., config["paths"]["local_repo_path"])
# Metadata: the class you just updated

overwrite = False  # Force refresh from Notion if True

# Paths
metadata_dir = os.path.join(data_dir, "00_Metadata")
os.makedirs(metadata_dir, exist_ok=True)
metadata_pickle_path = os.path.join(metadata_dir, "metadata_snapshot.pkl")

relations_map_dir = config["paths"].get("local_repo_path", metadata_dir)
os.makedirs(relations_map_dir, exist_ok=True)
relations_map_path = os.path.join(relations_map_dir, "relations_map.json")


def atomic_write_bytes(path: str, data: bytes):
    """Write bytes atomically to avoid partial/corrupt files."""
    dirpath = os.path.dirname(path)
    os.makedirs(dirpath, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=dirpath, delete=False) as tmp:
        tmp.write(data)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = tmp.name
    os.replace(tmp_path, path)


def atomic_write_text(path: str, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def load_all_tables(md_obj):
    """
    Convenience: pull all the DB/data source tables from a Metadata instance.
    Returns a dict so you can unpack if you want.
    """
    return {
        "deployment_db": md_obj.get_metadata("deployment_DB"),
        "logger_db": md_obj.get_metadata("logger_DB"),
        "recording_db": md_obj.get_metadata("recording_DB"),
        "animal_db": md_obj.get_metadata("animal_DB"),
        "dataset_db": md_obj.get_metadata("dataset_DB"),
        "procedure_db": md_obj.get_metadata("procedure_DB"),
        "observation_db": md_obj.get_metadata("observation_DB"),
        "collaborator_db": md_obj.get_metadata("collaborator_DB"),
        "location_db": md_obj.get_metadata("location_DB"),
        "montage_db": md_obj.get_metadata("montage_DB"),
        "signal_db": md_obj.get_metadata("signal_DB"),
        "attachment_db": md_obj.get_metadata("attachment_DB"),
        "originalchannel_db": md_obj.get_metadata("originalchannel_DB"),
        "standardizedchannel_db": md_obj.get_metadata("standardizedchannel_DB")
    }


def pickle_needs_refresh(path: str, max_age_days: int = 14) -> bool:
    if not os.path.exists(path):
        return True
    try:
        mtime = datetime.fromtimestamp(os.path.getmtime(path))
        return (datetime.now() - mtime) > timedelta(days=max_age_days)
    except Exception:
        # If anything is weird with the file, refresh.
        return True


# Decide whether to pull fresh data from Notion
needs_refresh = overwrite or pickle_needs_refresh(metadata_pickle_path, max_age_days=14)

if needs_refresh:
    # 1) Build a fresh Metadata instance (hits Notion and populates self.metadata etc.)
    metadata = Metadata()

    # 2) Recompute relations map (uses databases.retrieve schemas and relation fields)
    metadata.map_database_relations()
    relations_map = metadata.relations_map

    # 3) Save relations map atomically (so it’s never half-written)
    try:
        atomic_write_text(relations_map_path, json.dumps(relations_map, indent=4))
        print(f"Relations map saved at: {relations_map_path}")
    except Exception as e:
        print(f"[WARN] Failed to write relations_map.json: {e}")

    # 4) Strip live client / transient runtime caches before pickling
    metadata.notion = None
    if hasattr(metadata, "data_source_cache"):
        metadata.data_source_cache = {}

    # Optional: embed a tiny snapshot header for sanity
    snapshot_meta = {
        "notion_version": getattr(metadata, "notion_version", None),
        "created_at": datetime.now().isoformat(),
        "class": "Metadata",
    }
    payload = {"snapshot_meta": snapshot_meta, "metadata_obj": metadata}

    # 5) Snapshot full metadata object to disk atomically
    try:
        atomic_write_bytes(metadata_pickle_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
        print(f"[REFRESH] Metadata snapshot saved at: {metadata_pickle_path}")
    except Exception as e:
        print(f"[ERROR] Failed to write metadata snapshot: {e}")
        raise
else:
    # Load cached snapshot instead of hitting Notion
    print(f"[CACHE] Using existing metadata snapshot at: {metadata_pickle_path}")
    try:
        with open(metadata_pickle_path, "rb") as file:
            payload = pickle.load(file)
        # Backward-compat: support old format (raw Metadata pickled directly)
        if isinstance(payload, dict) and "metadata_obj" in payload:
            metadata = payload["metadata_obj"]
            snapshot_meta = payload.get("snapshot_meta", {})
        else:
            metadata = payload
            snapshot_meta = {}
        # Note: metadata.notion is None here (by design). We're only reading dfs, so it's fine.
    except Exception as e:
        print(f"[WARN] Cache unreadable ({e}). Falling back to fresh pull.")
        metadata = Metadata()
        metadata.map_database_relations()
        relations_map = metadata.relations_map
        try:
            atomic_write_text(relations_map_path, json.dumps(relations_map, indent=4))
        except Exception as ee:
            print(f"[WARN] Failed to write relations_map.json on fallback: {ee}")
        metadata.notion = None
        if hasattr(metadata, "data_source_cache"):
            metadata.data_source_cache = {}
        payload = {
            "snapshot_meta": {
                "notion_version": getattr(metadata, "notion_version", None),
                "created_at": datetime.now().isoformat(),
                "class": "Metadata",
            },
            "metadata_obj": metadata,
        }
        atomic_write_bytes(metadata_pickle_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
        print(f"[REFRESH] Metadata snapshot saved at: {metadata_pickle_path}")


# Expose each table for downstream code in this session
tables = load_all_tables(metadata)

deployment_db = tables["deployment_db"]
logger_db = tables["logger_db"]
recording_db = tables["recording_db"]
animal_db = tables["animal_db"]
dataset_db = tables["dataset_db"]
procedure_db = tables["procedure_db"]
observation_db = tables["observation_db"]
collaborator_db = tables["collaborator_db"]
location_db = tables["location_db"]
montage_db = tables["montage_db"]
signal_db = tables["signal_db"]
attachment_db = tables["attachment_db"]
originalchannel_db = tables["originalchannel_db"]
standardizedchannel_db = tables["standardizedchannel_db"]

# Optional: quick sanity print of row counts
try:
    counts = {k: (v.shape[0] if v is not None else 0) for k, v in tables.items()}
    print("[Metadata tables] row counts:", json.dumps(counts, indent=2))
except Exception:
    pass


In [ ]:
import json
import os

# Path to write JSON export
json_export_path = os.path.join(metadata_dir, "metadata_snapshot.json")

def df_to_records_safe(df):
    """Convert DataFrame to list of records, making all values JSON serializable."""
    return json.loads(df.to_json(orient="records", date_format="iso"))

# Convert all tables to dict of records
json_dict = {}
for name, df in tables.items():
    try:
        json_dict[name] = df_to_records_safe(df)
    except Exception as e:
        print(f"[WARN] Could not convert {name}: {e}")
        json_dict[name] = []

# Save as pretty JSON
with open(json_export_path, "w", encoding="utf-8") as f:
    json.dump(json_dict, f, indent=2)

print(f"✅ Metadata exported to {json_export_path}")


### Optional: Save metadata snapshot as pickle

In [ ]:
# Select dataset folder
dataset_folder = select_folder(data_dir, "Select a dataset folder:")

In [ ]:
deployment_folder = select_folder(dataset_folder, "Select a deployment folder:")

In [ ]:
# Extract deployment_id and animal_id from the folder name
match = re.match(r"(\d{4}-\d{2}-\d{2}_[a-z]{4}-\d{3})", os.path.basename(deployment_folder), re.IGNORECASE)
if match:
    deployment_id = match.group(1)  # Extract YYYY-MM-DD_animalID
    animal_id = deployment_id.split("_")[1]  # Extract animal ID
    print(f"✅ Extracted deployment ID: {deployment_id}, Animal ID: {animal_id}")
else:
    raise ValueError(f"❌ Unable to extract deployment ID from folder: {deployment_folder}")

## Read Files in Deployment Folder

Uses [`datareader`](../pyologger/load_data/datareader.py) class and its `read_files()` method to load and standardize data from a deployment folder, map onto standardized channel names, and save as a `data_pkl` object (instance of the datareader class).

In [ ]:
# Print extracted values for debugging
print(f"🐳 Deployment ID: {deployment_id}, Animal ID: {animal_id}")

deployment_info, loggers_used = metadata.extract_essential_metadata(deployment_id)

In [ ]:
import pytz
overwrite_essential_metadata = False

if overwrite_essential_metadata: # If you store your metadata differently, you can set this manually:
    # Deployment ID and Animal ID - this is important because it sets the start date and animal ID
    # Your dataset ID is the folder name that this deployment folder is in
    deployment_id = "2019-11-08_apfo-001"
    animal_id = "apfo-001"
    print(f"🔍 Manually setting essential metadata for Deployment ID: {deployment_id}")

    # Manually setting deployment metadata
    deployment_info = {
        "Deployment Date": "2019-11-08",
        "Deployment Latitude": -77.858933,
        "Deployment Longitude": 166.5139,
        "Time Zone": "Antarctica/McMurdo"
    }
    print(f"📍 Deployment Metadata: {deployment_info}")

    # Manually setting loggers used with Montage ID inside each entry
    loggers_used = [
        {"Logger ID": "CC-35", "Manufacturer": "CATS", "Montage ID": "cats-penguin-video-montage_V1"}
    ]
    print(f"📟 Loggers Used: {loggers_used}")

    # No separate montage_id list anymore, since it's stored in loggers_used

def list_available_timezones():
    """
    Prints all available time zones in pytz.
    """
    timezones = pytz.all_timezones
    print("\n🌍 Available Time Zones in pytz:\n")
    for tz in timezones:
        print(tz)

# List all available time zones
#list_available_timezones()

In [ ]:
# Step 4: Initialize DataReader with dataset folder, deployment ID, and optional data subfolder
data_pkl = DataReader(dataset_folder=dataset_folder, deployment_id=deployment_id, data_subfolder="01_raw-data", montage_path=montage_path)
# Step 5: Initialize config manager
param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)
param_manager.add_to_config("current_processing_step", "Processing Step 00: Data import pending.")

In [ ]:
loggers_used

In [ ]:
param_manager.export_config()

In [ ]:
overwrite_data = True
pkl_path = os.path.join(deployment_folder, "outputs", "data.pkl")
if os.path.exists(pkl_path) and not overwrite_data:
    with open(pkl_path, "rb") as f:
        data_pkl = pickle.load(f)
    print(f"📦 Loaded processed DataReader object from: {pkl_path}")
else:
    data_pkl = DataReader(
        dataset_folder=dataset_folder,
        deployment_id=deployment_id,
        data_subfolder="01_raw-data",
        montage_path=montage_path
    )
    param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)
    param_manager.add_to_config("current_processing_step", "Processing Step 00: Data import pending.")

    data_pkl.read_files(
        deployment_info=deployment_info,
        loggers_used=loggers_used,
        save_parq=False,
        save_netcdf=True
    )

In [ ]:
data_pkl.signal_data['o2_pressure']

In [ ]:
data_pkl.signal_info

In [ ]:
data_pkl.signal_data['pressure']

In [ ]:
data_pkl.event_data

In [ ]:
data_pkl.signal_info

In [ ]:
data_pkl.signal_data

In [ ]:
data_pkl.animal_info

In [ ]:
data_pkl.signal_info

In [ ]:
data_pkl.logger_info

In [ ]:
data_pkl.derived_info

In [ ]:
data_pkl.animal_info

In [ ]:
data_pkl.event_data

In [ ]:
data_pkl.signal_info

In [ ]:
import pandas as pd
from datetime import timedelta

# Get timezone
timezone = data_pkl.deployment_info.get("Time Zone", "UTC")
replace = True

# Load time settings
time_settings = param_manager.get_from_config(
    ["overlap_start_time", "overlap_end_time", "zoom_window_start_time", "zoom_window_end_time"],
    section="settings"
)

if time_settings and replace == False:
    print("Time settings present.")
# If any required time settings are missing, compute and update them
if not any(v is None for v in time_settings.values()) and replace == False:
    print("Time settings not empty.")
else:
    print("Adding timestamps to config.")
    zoom_time_window = 5  # minutes

    # Extract start and end times for all signals
    start_times = [df['datetime'].min() for df in data_pkl.signal_data.values()]
    end_times = [df['datetime'].max() for df in data_pkl.signal_data.values()]

    # Compute common start, end, and zoom window
    overlap_start_time = max(start_times)
    overlap_end_time = min(end_times)
    min_start_time = min(start_times)
    max_end_time = max(end_times)
    midpoint = overlap_start_time + (overlap_end_time - overlap_start_time) / 2
    zoom_window_start, zoom_window_end = midpoint - timedelta(minutes=zoom_time_window / 2), midpoint + timedelta(minutes=zoom_time_window / 2)

    # Update settings
    time_settings = {
        "overlap_start_time": str(overlap_start_time),
        "overlap_end_time": str(overlap_end_time),
        "zoom_window_start_time": str(zoom_window_start),
        "zoom_window_end_time": str(zoom_window_end),
    }
    param_manager.add_to_config(entries=time_settings, section="settings")

if any(v is None for v in time_settings.values()):
    print("YES")
time_settings

print(f"earliest logger start: {min_start_time}")
print(f"latest logger end: {max_end_time}")

In [ ]:
time_settings

In [ ]:
print(f"earliest logger start: {min_start_time}")
print(f"latest logger end: {max_end_time}")
overlap_end_time-overlap_start_time

In [ ]:
data_pkl.signal_data['o2_pressure']

In [ ]:
# Plot arterial pO2 vs datetime
import plotly.express as px

df_o2 = data_pkl.signal_data['o2_pressure']

# If datetime is the index, move it to a column for plotting
if 'datetime' not in df_o2.columns and isinstance(df_o2.index, pd.DatetimeIndex):
    df_o2 = df_o2.reset_index().rename(columns={df_o2.index.name or 'index': 'datetime'})

# Ensure datetime column is datetime dtype
df_o2['datetime'] = pd.to_datetime(df_o2['datetime'])

fig = px.line(
    df_o2,
    x='datetime',
    y='o2_pressure_arterial',
    title=f"{deployment_id}: arterial pO2 over time",
    labels={'datetime': 'Datetime', 'o2_pressure_arterial': 'pO2 arterial'}
)
fig.update_layout(hovermode='x unified')
fig.show()

In [ ]:
data_pkl.signal_data['pressure']

In [ ]:
start = pd.Timestamp(time_settings['overlap_start_time'])# - timedelta(hours = 30)
end = pd.Timestamp(time_settings['overlap_end_time'])# + timedelta(hours=30)

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    # signals = ['pressure', 'o2_pressure', 'light', 'temperature_int', 'temperature_ext'],
    # signals=['ecg', 'pressure'], # 'eeg', 'accelerometer', 'accelerometer2','prh'],
    time_range=(start, end),
    note_annotations={"dive": {"signal": "depth", "symbol": "triangle-down", "color": "blue"}},
    state_annotations={"dive": {"signal": "depth", "color": "rgba(150, 150, 150, 0.3)"}},
    zoom_range_selector_channel='ecg',
    color_mapping_path=color_mapping_path,
    target_sampling_rate=25
)
fig.show_dash(mode="inline")

In [ ]:
data_pkl.event_data

In [ ]:
# optional save
pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')
with open(pkl_path, "wb") as file:
    pickle.dump(data_pkl, file)

In [ ]:
import xarray as xr

# Step 8: Update processing step
param_manager.add_to_config("current_processing_step", "Processing Step 00: Data imported.")

# Step 9: Open NetCDF file
netcdf_path = os.path.join(deployment_folder, "outputs", f'{deployment_id}_00_processed.nc')
if os.path.exists(netcdf_path):
    data = xr.open_dataset(netcdf_path)
    print(f"📊 NetCDF file loaded: {netcdf_path}")
else:
    print(f"⚠ NetCDF file not found at {netcdf_path}.")

data

In [ ]:
# Check if selected start and end times exist in the config file
truncate_times = param_manager.get_from_config(
    ["selected_start_time", "selected_end_time"],
    section="settings"
)

truncate_times

In [ ]:
if not any(v is None for v in truncate_times.values()):
    print("Truncating with provided cropping times.")
    # Update overlap window with selected range
    OVERLAP_START_TIME = pd.Timestamp(truncate_times['selected_start_time']).tz_convert(timezone)
    OVERLAP_END_TIME = pd.Timestamp(truncate_times['selected_end_time']).tz_convert(timezone)

    # Truncate signal data
    for signal, df in data_pkl.signal_data.items():
        # Truncate based on selected time range
        truncated_df = df[(df.iloc[:, 0] >= OVERLAP_START_TIME) & (df.iloc[:, 0] <= OVERLAP_END_TIME)].copy()
        data_pkl.signal_data[signal] = truncated_df  # Save truncated version to new variable

    # Recalculate Zoom Window (5-minute window in the middle)
    midpoint = OVERLAP_START_TIME + (OVERLAP_END_TIME - OVERLAP_START_TIME) / 2
    ZOOM_WINDOW_START_TIME = midpoint - timedelta(minutes=2.5)
    ZOOM_WINDOW_END_TIME = midpoint + timedelta(minutes=2.5)

    # Save new time settings
    time_settings_update = {
        "overlap_start_time": str(OVERLAP_START_TIME),
        "overlap_end_time": str(OVERLAP_END_TIME),
        "zoom_window_start_time": str(ZOOM_WINDOW_START_TIME),
        "zoom_window_end_time": str(ZOOM_WINDOW_END_TIME)
    }
    param_manager.add_to_config(entries=time_settings_update, section="settings")

    pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')
    with open(pkl_path, "wb") as file:
        pickle.dump(data_pkl, file)

In [ ]:
fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    time_range=(OVERLAP_START_TIME, OVERLAP_END_TIME),
    zoom_start_time= ZOOM_WINDOW_START_TIME,
    zoom_end_time= ZOOM_WINDOW_END_TIME,
    note_annotations={"dive": {"signal": "depth", "symbol": "triangle-down", "color": "blue"}},
    state_annotations={"dive": {"signal": "depth", "color": "rgba(150, 150, 150, 0.3)"}},
    color_mapping_path=color_mapping_path,
    target_sampling_rate=1
)
fig.show_dash(mode="inline")

## Inspect data

In [ ]:
# Load the data_reader object from the pickle file
pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')

with open(pkl_path, 'rb') as file:
    data_pkl = pickle.load(file)

for logger_id, info in data_pkl.logger_info.items():
    sampling_frequency = info.get('datetime_metadata', {}).get('fs', None)
    if sampling_frequency is not None:
        # Format the sampling frequency to 5 significant digits
        print(f"Sampling frequency for {logger_id}: {sampling_frequency} Hz")
    else:
        print(f"No sampling frequency available for {logger_id}")